# Lecture 12: Non-Linear Programming - Principles

---

```{note}
Lectures 1–11 built a complete toolkit for **linear** optimisation: the objective function and every constraint were straight lines (or hyperplanes). Many real transportation problems — signal timing, congestion pricing, facility siting, vehicle routing — are not linear: costs rise non-proportionally with volume, constraints are products or ratios of decision variables, and the feasible region is no longer a convex polygon. This lecture introduces the landscape of **Non-Linear Programming (NLP)**: what makes a problem non-linear, how non-linearity changes the geometry and the solution strategy, and which families of NLP methods we will study in Lectures 13–22.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Recognise the structural differences between linear and non-linear programs, and classify NLP instances by problem type (unconstrained/constrained, convex/non-convex).
2. Identify local and global optima geometrically and analytically, and explain why LP methods fail in non-linear settings.
3. Map real transportation optimisation problems to the appropriate NLP framework and solver family covered in this module.

**Prerequisites**: LP Formulation (Lecture 3), Graphical Solution Method (Lecture 4), Simplex Method (Lecture 7), Duality (Lecture 9)

**Estimated time**: 90 minutes (including in-class exercises)

---

## Why Non-Linear Programming?

Every LP we solved in Lectures 3–11 rested on a silent assumption: the relationships between decision variables are proportional and additive. Doubling the number of buses doubles the cost. Sending twice as much freight doubles fuel use. Adding one more worker doubles output. In practice this linearity assumption breaks down in three common ways.

**Diminishing Effects:** Every additional vehicle on the Sardar Patel Road road reduces traffic efficiency polynomially.

**Fixed Assets:** Opening a new freight terminal in Chennai incurs a fixed cost regardless of volume.

**Interaction Effects.** The fuel consumption of a truck on a hilly highway is a function of speed, acceleration, and load.

### Example 1

CMRL wants to set green-time splits $(x_1, x_2)$ at a junction to minimise total vehicle delay. Webster's delay formula gives:

$$D_i(x_i) = \frac{C(1 - x_i/C)^2}{2(1 - \lambda_i x_i / C)} + \frac{\lambda_i^2}{2\mu_i(1-\lambda_i)}$$

where $C$ is cycle length, $\lambda_i$ arrival rate, $\mu_i$ saturation flow.

### Example 2

A logistics firm wants to locate a transshipment hub at coordinates $(x, y)$ to minimise total weighted distance to $n$ customer locations $(a_k, b_k)$ with demand weights $w_k$:

$$\min_{x,y} \quad f(x,y) = \sum_{k=1}^{n} w_k \sqrt{(x - a_k)^2 + (y - b_k)^2}$$

### Example 3

The Beckmann formulation of Wardrop's user equilibrium assigns traffic volumes $v_a$ to roads to solve:

$$\min_{v} \quad \sum_{a \in \mathcal{A}} \int_0^{v_a} t_a(u)\, du$$

where $t_a(\cdot)$ is the BPR travel-time function. 

> **Key question**: once we leave the land of linear programs, what tools replace the Simplex method?

---

## Standard Form

A **Non-Linear Program** is an optimisation problem of the form:

$$
\begin{aligned}
\min_{x} \quad & f(x) \\
\text{subject to} \quad & g_i(x) \leq 0, \qquad i = 1, \ldots, m \\
& x \in \mathbb{R}^n
\end{aligned}
$$

where at least one of $f$ or $g_i$ is **non-linear** in $x$. Equality constraints $g_j(x) = 0$ are handled exactly as in LP — converted to a pair $g_j(x) \leq 0$ and $-g_j(x) \leq 0$ — so no separate equality row is needed.

**Notation:**

| Symbol | Meaning |
|--------|---------|
| $x = (x_1, \ldots, x_n)^\top$ | Vector of $n$ decision variables |
| $f : \mathbb{R}^n \to \mathbb{R}$ | Objective function |
| $g_i : \mathbb{R}^n \to \mathbb{R}$ | Constraint function ($\leq 0$ form; equalities handled by conversion) |
| $\nabla f(x)$ | Gradient of $f$ at $x$ — vector of first partial derivatives |
| $\nabla^2 f(x)$ | Hessian of $f$ at $x$ — matrix of second partial derivatives |
| $\mathcal{F}$ | Feasible region: $\{x : g_i(x) \leq 0 \; \forall\, i\}$ |

```{note}
The LP is a special case where all functions are linear. In compact matrix–vector notation ($\mathbf{c} \in \mathbb{R}^n$, $\mathbf{A} \in \mathbb{R}^{m \times n}$, $\mathbf{b} \in \mathbb{R}^m$):

$$\max_{\mathbf{x} \geq \mathbf{0}} \quad \mathbf{c}^\top \mathbf{x} \qquad \text{subject to} \qquad \mathbf{A}\mathbf{x} \leq \mathbf{b}$$

This is the NLP general form specialised to $f(\mathbf{x}) = -\mathbf{c}^\top \mathbf{x}$ (linear objective, negated for $\min$) and $g_i(\mathbf{x}) = \mathbf{A}_i \mathbf{x} - b_i \leq 0$ (linear constraints, where $\mathbf{A}_i$ is the $i$-th row of $\mathbf{A}$). The familiar LP optimality conditions (shadow prices, reduced costs, duality) become special cases of the NLP conditions (KKT multipliers, Lagrangian duality) we derive in the later lectures.
```

---

## Taxonomy

### Constrained vs Unconstrained Optimisation

**Unconstrained Optimization**: no constraints; the feasible region is all of $\mathbb{R}^n$.

$$\min_{x \in \mathbb{R}^n} \; f(x)$$

A necessary condition for $x^*$ to be a local minimum is that the gradient vanishes:

$$\nabla f(x^*) = \mathbf{0} \qquad \text{(First-Order Necessary Condition)}$$

A point satisfying this condition is called a stationary point - it may be a minimum, maximum, or saddle point. To confirm a minimum, we require the second-order sufficient condition:

$$\nabla^2 f(x^*) \succ 0 \qquad \text{(Second-Order Sufficient Condition)}$$

that is, the Hessian $\nabla^2 f(x^*)$ must be positive definite — all eigenvalues strictly positive.

**Constrained Optimization**: at least one constraint is active, restricting $\mathcal{F} \subsetneq \mathbb{R}^n$.

$$
\begin{aligned}
\min_{x \in \mathcal{F}} \quad & f(x) \\
\text{subject to} \quad & g_i(x) \leq 0, \qquad i = 1, \ldots, m \\
                        & x \in \mathbb{R}^n
\end{aligned}
$$

Here, stationarity of $f$ alone is not sufficient: the optimum may lie on the boundary of $\mathcal{F}$, where the constraint is active. Optimality is instead characterised by the Lagrangian:

$$\mathcal{L}(x, \lambda) = f(x) + \sum_{i=1}^{m} \lambda_i\, g_i(x)$$

The Karush–Kuhn–Tucker (KKT) conditions — derived fully in Lecture 15 — require $\nabla_x \mathcal{L}(x^*, \lambda^*) = \mathbf{0}$ together with complementary slackness $\lambda_i g_i(x^*) = 0$ and dual feasibility $\lambda_i \geq 0$. They extend the LP optimality condition (all reduced costs $\geq 0$) to the non-linear, constrained setting.

### Convex vs Non-Convex Program

**Convex Program**: $f$ is convex, every $g_i$ is convex, and $\mathcal{F}$ is therefore a convex set.

A set $\mathcal{F}$ is convex if the line segment joining any two of its points lies entirely within it:

$$x_1, x_2 \in \mathcal{F},\; \lambda \in [0,1] \implies \lambda x_1 + (1-\lambda)x_2 \in \mathcal{F}$$

A function $f$ is convex if it lies at or below the line joining any two of its points:

$$f\!\left(\lambda x_1 + (1-\lambda)x_2\right) \leq \lambda f(x_1) + (1-\lambda) f(x_2) \quad \forall\, x_1, x_2,\; \lambda \in [0,1]$$

Notably, every local minimum of a convex program is also a global minimum. This guarantee means gradient-based methods (iterative descent) — which converge to a point where $\nabla f = \mathbf{0}$ — are sufficient to find the globally optimal solution.

**Non-convex Program**: $f$ or $\mathcal{F}$ (or both) are non-convex. A local minimum need not be a global minimum, and hence gradient-based solvers may converge to a local optima.

```{note}

A point $x^*$ is a local minimum of $f$ on $\mathcal{F}$ if there exists $\epsilon > 0$ such that:

$$f(x^*) \leq f(x) \quad \forall\, x \in \mathcal{F} \text{ with } \|x - x^*\| < \epsilon$$

A point $x^*$ is the global minimum*of $f$ on $\mathcal{F}$ if:

$$f(x^*) \leq f(x) \quad \forall\, x \in \mathcal{F}$$
```
---

## Why LP Methods Fail for NLP

The Simplex method's power comes from three facts that are unique to LP:

1. Convex Polytope Feasible Region: Every corner (vertex) is a candidate optimal solution, and the Simplex method hops between adjacent vertices along improving edges.

2. Linear Objective: The gradient $\nabla f = \mathbf{c}$ is constant — it never changes direction, so the method always knows which way is downhill.

3. Global Optimum: LP's strong duality theorem guarantees that when Simplex stops (all reduced costs $\geq 0$), the current vertex is globally optimal — no other point anywhere in $\mathcal{F}$ can do better.

None of these three properties holds for a general NLP, wherein $\mathcal{F}$ can be non-convex with $\nabla f(x)$ that changes with $x$ rendering potenital local optimals or saddle points.

---

## In-Class Exercises

### Exercise 1

BMTC is planning bus frequencies $x_1, x_2$ (buses/hour) on two Bengaluru corridors. Passenger demand on each corridor increases with service frequency, but at a diminishing rate. Consequently, the total revenue is given by: $R(x_1, x_2) = 800\sqrt{x_1} + 600\sqrt{x_2}$. Notably, BMTC has a fleet availability constraint that limits the total number of buses deployed to 20 per hour. In addition, to maintain a minimum level of service, each corridor must receive at least 3 buses per hour. Given, operating cost of ₹1,200 per bus-hour, 

1. Write the NLP in standard form.
2. Classify the problem — constrained or unconstrained, linear or non-linear, convex or non-convex.

### Exercise 2

A Delhivery cross-dock facility in Guwahati processes freight arriving at rate $\lambda$ (tonnes/hour). The variable processing cost increases with throughput and is given by: $C(\lambda) = 50\lambda + 2\lambda^2$. In addition, the facility incurs a fixed operating cost of ₹8,000 per hour. The facility earns revenue of ₹200 for every tonne of freight processed. To satisfy the service-level agreement, the facility must process at least 30 tonnes per hour, however, due to equipment capacity constraints, throughput cannot exceed 80 tonnes per hour.

1. Write the NLP in standard form.
2. Classify the problem — constrained or unconstrained, linear or non-linear, convex or non-convex.

---

## Take-Away Exercises

### Exercise 1

A NHAI corridor study estimates travel time on Chandigarh-Ambala link using the BPR function:

$$t(v) = 15 \left[1 + 0.15 \left(\frac{v}{1{,}200}\right)^4\right] \quad \text{(minutes)}$$

where $v$ is traffic volume (vehicles/hour) and capacity is $c = 1{,}200$ veh/hr.

1. Compute $t(600)$, $t(1{,}200)$, and $t(2{,}400)$. Comment on how travel time responds to volume near and above capacity.
2. Differentiate $t(v)$ with respect to $v$ and evaluate $t'(1{,}200)$. Interpret the result: what does this derivative tell NHAI about marginal congestion at capacity?
3. Is $t(v)$ a convex function of $v$ for $v \geq 0$? Justify by computing $t''(v)$ and checking its sign.

### Exercise 2

The traffic volume $v$ on a Mumbai expressway link is assigned to minimise the Beckmann objective:

$$B(v) = \int_0^{v} t_0 \left[1 + 0.15\left(\frac{u}{c}\right)^4\right] du $$

with $t_0 = 20$ min, $c = 2{,}000$ veh/hr, and the constraint $0 \leq v \leq 3{,}000$.

1. Is $B(v)$ convex on $[0, 3000]$? Show using $B''(v)$.
2. The demand constraint requires $v \geq 1{,}500$. Write the constrained NLP in standard form ($g(v) \leq 0$).

---

## Further Reading

- Bazaraa, M.S., Sherali, H.D., and Shetty, C.M. (2006). *Nonlinear Programming: Theory and Algorithms* (3rd ed.). Wiley — Chapters 1–2 (problem classes, optimality conditions overview).
- Nocedal, J. and Wright, S.J. (2006). *Numerical Optimization* (2nd ed.). Springer — Chapter 1 (introduction), Chapter 2 (fundamentals of unconstrained optimisation).
- Wardrop, J.G. (1952). Some theoretical aspects of road traffic research. *Proceedings of the Institute of Civil Engineers*, 1(3), 325–362. — Origin of the traffic equilibrium conditions.
- BPR (1964). *Traffic Assignment Manual*. US Bureau of Public Roads, Washington DC. — Source of the BPR travel-time function.
- SciPy documentation: `scipy.optimize.minimize` — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)